# Stage 05 — Held-out evaluation set

**Track A (Buse) · Stage 5 of 10**

| | |
|---|---|
| **Input** | The domain brief from stage 00, the chunk file from stage 02 |
| **Output** | `eval/datasets/queries_B.jsonl`, `eval/datasets/qrels_B.jsonl` |
| **Promotes to** | Nothing. These files *are* the deliverable, and Sude's gate consumes them unmodified. |

## Why this is the most valuable thing you build

Every number in the report, every promotion decision, and the entire DPO training
signal trace back to this file. A sloppy eval set produces confident, meaningless
numbers, and nobody downstream can tell.

It is also the one artefact that must not move once the gate is live. Changing
labels changes the threshold's meaning retroactively.

## Design choices

| Choice | Picked | Alternatives | Why |
|---|---|---|---|
| Query source | Written by you from the brief's question shapes | Sampled from real logs, LLM-generated | You have no logs yet. LLM-generated queries quietly inherit the retriever's own blind spots, which makes the eval flatter than reality. |
| Query count | 40-60 | 20, 200 | 40 is the floor for a stable nDCG. The gate refuses to run below it. |
| Labels | Graded, 0-3 | Binary relevant/not | nDCG needs grades to distinguish "the exact paragraph" from "the right paper, wrong section". That distinction is exactly what a reranker learns. |
| Who labels | You, by hand | LLM judge | Sude's judge is validated *against* these labels in her track. If the judge produces them, the validation is circular and worthless. |
| Pooling | Label the union of the top 20 from each retrieval arm | Label everything, label only top 5 | Labelling the whole corpus per query is impossible. Pooling across arms is the standard construction and it avoids favouring the arm you happened to build first. |
| Split | All held out, none used for training | Train/test split | Your DPO pairs in stage 07 come from *unlabelled* queries scored by the judge. These labelled queries stay clean, and the leakage test enforces it. |

### Grade scale, write this down and stick to it

| Grade | Meaning |
|---|---|
| 3 | Directly answers the question. You could cite this chunk alone. |
| 2 | Contains part of the answer, or the answer without its qualifier. |
| 1 | Right topic, right paper, does not answer the question. |
| 0 | Not relevant. |

The 2 versus 1 boundary is where annotator drift happens. Write your own tie-breaking
rule here in one sentence and follow it, so a label made on day three matches day one.

In [ ]:
from _nbsetup_B import REPO, load_cfg, resolve
import json
from pathlib import Path

icfg = load_cfg("ingestion")
chunks = [json.loads(l) for l in (resolve(icfg["corpus"]["processed_dir"]) / "chunks.jsonl")
          .read_text(encoding="utf-8").splitlines() if l.strip()]
by_id = {c["chunk_id"]: c for c in chunks}
print(len(chunks), "chunks available to label")

In [ ]:
# Queries. `shape` ties each one back to a question shape in the domain brief, so you
# can prove coverage rather than assert it. Aim for balance across shapes.
QUERIES = [
    dict(query_id="q001", shape="TODO shape 1", query="TODO: a real question",
         notes="why this is answerable from the corpus"),
    dict(query_id="q002", shape="TODO shape 2", query="TODO", notes=""),
]

import collections, pandas as pd
print(collections.Counter(q["shape"] for q in QUERIES))
pd.DataFrame(QUERIES)

In [ ]:
# Pooling: build the candidate pool per query from both arms, then label the pool.
# Import the search functions you promoted from notebook 04.
from research_assistant.retrieval.hybrid_B import hybrid_search  # after you promote stage 04

POOL_N = 20

def build_pool(query):
    return [c["chunk_id"] for c in hybrid_search(query, candidate_k=POOL_N)]

pools = {q["query_id"]: build_pool(q["query"]) for q in QUERIES}
print({k: len(v) for k, v in pools.items()})

In [ ]:
# Labelling helper. Prints one chunk at a time so you grade with the text in front of
# you rather than from memory. Grades go into LABELS as you work.
LABELS = {}   # (query_id, chunk_id) -> grade

def show(query_id, chunk_id):
    q = next(x for x in QUERIES if x["query_id"] == query_id)
    c = by_id[chunk_id]
    print("Q:", q["query"])
    print(f"{c['title']} / {c['section']} / p{c['page_start']}-{c['page_end']}")
    print("-" * 70)
    print(c["text"][:1500])
    print("-" * 70)
    print("grade 3=answers  2=partial  1=on-topic  0=irrelevant")

# show("q001", pools["q001"][0])
# LABELS[("q001", pools["q001"][0])] = 3

In [ ]:
# Write the two files. Schema is frozen: Sude's gate reads it without modification,
# and it is mirrored in src/research_assistant/contracts/eval_dataset_J.py.
qpath = REPO / "eval" / "datasets" / "queries_B.jsonl"
rpath = REPO / "eval" / "datasets" / "qrels_B.jsonl"

with qpath.open("w", encoding="utf-8") as f:
    for q in QUERIES:
        f.write(json.dumps(dict(query_id=q["query_id"], query=q["query"],
                                shape=q["shape"], notes=q.get("notes", "")),
                           ensure_ascii=False) + "\n")

with rpath.open("w", encoding="utf-8") as f:
    for (qid, cid), grade in sorted(LABELS.items()):
        c = by_id[cid]
        f.write(json.dumps(dict(query_id=qid, chunk_id=cid, grade=grade,
                                paper_id=c["paper_id"], page_start=c["page_start"],
                                page_end=c["page_end"]), ensure_ascii=False) + "\n")

print("queries:", len(QUERIES), "->", qpath)
print("labels:", len(LABELS), "->", rpath)

## Exit checks

- [ ] At least 40 queries, spread across every question shape in the brief.
- [ ] Every query has at least one grade-3 chunk. A query with no correct answer in
      the corpus measures nothing and drags every metric down uniformly.
- [ ] You re-labelled 10 chunks a day later and agreed with yourself on at least 8.
      If not, your grade boundary is not written clearly enough. Fix the rule, relabel.
- [ ] Sude can load both files with the contract model and no local edits.
- [ ] These queries never enter the pair builder in stage 07. The leakage test
      `tests/test_no_eval_leakage_J.py` enforces this. Make it fail once on purpose,
      so you know it works.